要運行以下筆記本，如果尚未完成，您需要部署一個以 `text-embedding-ada-002` 為基礎模型的模型，並在 .env 文件中將部署名稱設置為 `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


接下來，我們將把 Embedding 索引載入到 Pandas Dataframe 中。Embedding 索引存放在一個名為 `embedding_index_3m.json` 的 JSON 檔案內。Embedding 索引包含截至 2023 年 10 月底所有 YouTube 轉錄文字的 Embeddings。


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

接著，我們將建立一個名為 `get_videos` 的函數，用來在 Embedding 索引中搜尋查詢。該函數會回傳與查詢最相似的前 5 個影片。函數的運作步驟如下：

1. 首先，建立 Embedding 索引的副本。
2. 接著，使用 OpenAI Embedding API 計算查詢的 Embedding。
3. 然後，在 Embedding 索引中新增一個名為 `similarity` 的欄位。`similarity` 欄位包含查詢 Embedding 與每個影片片段的 Embedding 之間的餘弦相似度。
4. 接著，根據 `similarity` 欄位過濾 Embedding 索引。僅保留餘弦相似度大於或等於 0.75 的影片。
5. 最後，根據 `similarity` 欄位排序 Embedding 索引，並回傳前 5 個影片。


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

呢個函數好簡單，佢淨係打印出搜索查詢嘅結果。


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. 首先，將嵌入索引加載到 Pandas 資料框中。
2. 接着，系統會提示用戶輸入查詢。
3. 然後調用 `get_videos` 函數在嵌入索引中搜索查詢條目。
4. 最後調用 `display_results` 函數向用戶顯示結果。
5. 接著系統會提示用戶輸入另一個查詢。此過程會一直持續，直到用戶輸入 `exit`。

![](../../../../translated_images/zh-MO/notebook-search.1e320b9c7fcbb0bc.webp)

系統會提示你輸入查詢。輸入查詢後按下 Enter。應用程式會返回與查詢相關的視頻列表。應用程式還會返回一個連結，指向視頻中該問題答案所在的位置。

這裡有一些可以試試的查詢：

- 什麼是 Azure 機器學習？
- 卷積神經網絡是如何運作的？
- 什麼是神經網絡？
- 我可以在 Azure 機器學習中使用 Jupyter 筆記本嗎？
- 什麼是 ONNX？


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
